# AITM Red-Teaming — Colab Environment (Reusable via Google Drive)

**GPU required:** A100 40 GB (Colab Pro) for 4-bit · A100 80 GB (Colab Pro+) for BF16  
**Model:** `google/gemma-4-31b-it`  
**Key idea:** uv venv + model weights live on Drive → no reinstall on reconnect

---
### First-run checklist
1. Runtime → Change runtime type → **A100 GPU**
2. Set `HF_TOKEN` below (Hugging Face token with Gemma access)
3. Set `NGROK_TOKEN` below (free at ngrok.com)
4. Run all cells in order

### Subsequent sessions
- Cells 1-3: mount Drive & restore uv/venv (seconds, no download)
- Cell 4: set tokens
- Cell 5: start vLLM (model loads from Drive cache, ~2-3 min)
- Cell 6: get ngrok URL → paste into benchmark `--url`

In [ ]:
# ── CELL 1: Mount Google Drive ────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/aitm-env'
!mkdir -p {DRIVE}/hf_cache

In [ ]:
# ── CELL 2: Install / restore uv from Drive ───────────────────────────────────
import os, subprocess

UV_CACHED = f'{DRIVE}/uv'
UV_BIN    = '/usr/local/bin/uv'

if os.path.exists(UV_CACHED):
    subprocess.run(f'cp {UV_CACHED} {UV_BIN} && chmod +x {UV_BIN}', shell=True)
    print('uv restored from Drive')
else:
    subprocess.run('curl -LsSf https://astral.sh/uv/install.sh | sh', shell=True)
    subprocess.run(f'cp ~/.local/bin/uv {UV_CACHED}', shell=True)
    print('uv installed and cached to Drive')

!uv --version

In [ ]:
# ── CELL 3: Create / restore venv on Drive ────────────────────────────────────
VENV = f'{DRIVE}/.venv'

if not os.path.exists(f'{VENV}/bin/python'):
    print('Creating venv on Drive (first time only)...')
    !uv venv {VENV} --python 3.12
    !uv pip install --python {VENV}/bin/python \
        vllm \
        pyngrok \
        openai \
        huggingface_hub
    print('venv created and saved to Drive')
else:
    print('venv restored from Drive — no reinstall needed')

PYTHON = f'{VENV}/bin/python'
!{PYTHON} --version

In [ ]:
# ── CELL 4: Tokens (fill these in each session) ───────────────────────────────
import os

HF_TOKEN    = ''   # huggingface.co/settings/tokens  (needs Gemma access)
NGROK_TOKEN = ''   # dashboard.ngrok.com/get-started/your-authtoken

os.environ['HF_TOKEN']                = HF_TOKEN
os.environ['HUGGING_FACE_HUB_TOKEN']  = HF_TOKEN
os.environ['HF_HOME']                 = f'{DRIVE}/hf_cache'
os.environ['HUGGINGFACE_HUB_CACHE']   = f'{DRIVE}/hf_cache'

print('Tokens set. HF model cache →', os.environ['HF_HOME'])

In [ ]:
# ── CELL 5: Launch vLLM server ────────────────────────────────────────────────
# First run: downloads model weights to Drive/hf_cache (~60 GB, ~15-30 min)
# Subsequent runs: loads from Drive cache (~2-3 min)
import subprocess, time

MODEL = 'google/gemma-4-31b-it'
PORT  = 8000

proc = subprocess.Popen(
    [
        PYTHON, '-m', 'vllm.entrypoints.openai.api_server',
        '--model',                  MODEL,
        '--max-model-len',          '8192',
        '--gpu-memory-utilization', '0.90',
        '--tensor-parallel-size',   '1',
        '--port',                   str(PORT),
    ],
    stdout=open(f'{DRIVE}/vllm.log', 'w'),
    stderr=subprocess.STDOUT,
)

print(f'vLLM PID {proc.pid} — waiting for server to be ready...')
print(f'Logs: {DRIVE}/vllm.log')

import urllib.request
for _ in range(120):
    try:
        urllib.request.urlopen(f'http://localhost:{PORT}/health')
        print('vLLM ready!')
        break
    except:
        time.sleep(5)
else:
    print('Timeout — check logs at', f'{DRIVE}/vllm.log')

In [ ]:
# ── CELL 6: Expose via ngrok → get public URL ─────────────────────────────────
from pyngrok import ngrok, conf

conf.get_default().auth_token = NGROK_TOKEN
tunnel = ngrok.connect(PORT, bind_tls=True)
PUBLIC_URL = tunnel.public_url

print(f'\n{'='*60}')
print(f'vLLM endpoint: {PUBLIC_URL}/v1')
print(f'Model:         {MODEL}')
print(f'{'='*60}')
print(f'\nRun benchmark with:')
print(f'  python benchmark.py --adapter autogen --topo chain \\')
print(f'    --dataset mbpp --n_samples 20 \\')
print(f'    --url "{PUBLIC_URL}/v1"')

In [ ]:
# ── CELL 7 (optional): Clone repo inside Colab & run benchmark directly ───────
REPO = 'https://github.com/highphysicist/aitm-red-teaming-mas.git'
BRANCH = 'gemma-colab'

!git clone --branch {BRANCH} {REPO} /content/aitm 2>&1 | tail -3
%cd /content/aitm

# Install project deps into Drive venv
!uv pip install --python {PYTHON} -r requirements.txt -q

# Set the vLLM URL for this session
import subprocess
result = subprocess.run(
    [
        PYTHON, 'benchmark.py',
        '--adapter', 'autogen', '--topo', 'chain',
        '--dataset', 'mbpp', '--n_samples', '5',
        '--url', f'{PUBLIC_URL}/v1',
    ],
    capture_output=False
)